# Description

In this notebook, prepare the data for the experiment with sclability wrt input dimensions.

In [1]:
# scalability_experiment_data.py

import numpy as np
import sympy as sp
import h5py

from config.scalability_config import DataCFG

np.random.seed(DataCFG.seed)


def sample_signed_uniform(low=0.5, high=3.0, decimals=2):
    mag = np.random.uniform(low, high)
    sgn = -1.0 if np.random.rand() < 0.5 else 1.0
    return float(np.round(sgn * mag, decimals))


def sample_poly3_coeffs(k: int):
    """
    Dense total-degree-3 polynomial in k variables using:
      c0
      lin:            x_i
      quad (i<=j):    x_i x_j  (includes squares)
      cub  (i<=j<=l): x_i x_j x_l (includes x_i^3, x_i^2 x_j, etc.)
    """
    c0 = sample_signed_uniform()

    lin = np.array([sample_signed_uniform() for _ in range(k)], dtype=float)

    iu2 = np.triu_indices(k, k=0)  # includes diagonal
    quad = np.array([sample_signed_uniform() for _ in range(iu2[0].shape[0])], dtype=float)

    # triples with repetition i<=j<=l
    triples_i = []
    triples_j = []
    triples_l = []
    for i in range(k):
        for j in range(i, k):
            for l in range(j, k):
                triples_i.append(i)
                triples_j.append(j)
                triples_l.append(l)
    triples_i = np.array(triples_i, dtype=np.int32)
    triples_j = np.array(triples_j, dtype=np.int32)
    triples_l = np.array(triples_l, dtype=np.int32)

    cub = np.array([sample_signed_uniform() for _ in range(triples_i.shape[0])], dtype=float)

    return (c0, lin, quad, cub, (triples_i, triples_j, triples_l))


def eval_poly3(xk: np.ndarray, c0: float, lin: np.ndarray, quad: np.ndarray, cub: np.ndarray, tri_idx):
    y = c0 + xk @ lin

    k = xk.shape[1]

    iu2 = np.triu_indices(k, k=0)
    y = y + np.sum((xk[:, iu2[0]] * xk[:, iu2[1]]) * quad[None, :], axis=1)

    ti, tj, tl = tri_idx
    y = y + np.sum((xk[:, ti] * xk[:, tj] * xk[:, tl]) * cub[None, :], axis=1)

    return y


def poly3_to_sympy(c0, lin, quad, cub, tri_idx, xs):
    expr = sp.Float(c0)

    for j, a in enumerate(lin):
        expr += sp.Float(float(a)) * xs[j]

    k = len(xs)
    iu2 = np.triu_indices(k, k=0)
    for t, (i, j) in enumerate(zip(iu2[0], iu2[1])):
        expr += sp.Float(float(quad[t])) * xs[i] * xs[j]

    ti, tj, tl = tri_idx
    for t in range(len(cub)):
        i = int(ti[t])
        j = int(tj[t])
        l = int(tl[t])
        expr += sp.Float(float(cub[t])) * xs[i] * xs[j] * xs[l]

    return sp.simplify(expr)


def expr_to_sympy(expr):
    k = expr["k"]
    xs = sp.symbols("x0:" + str(k), real=True)
    p = poly3_to_sympy(*expr["p"], xs)
    y = sp.simplify(p)
    return xs, p, y


def generate_expression(k=2):
    p_coeffs = sample_poly3_coeffs(k)
    return {"kind": "P3", "k": k, "p": p_coeffs}


def eval_expression(expr, x):
    k = expr["k"]
    xk = x[:, :k]
    c0, lin, quad, cub, tri_idx = expr["p"]
    y = eval_poly3(xk, c0, lin, quad, cub, tri_idx)
    return y


def sample_dataset_uniform(expr, d, n, lo, hi):
    xs = []
    ys = []
    need = n
    factor = 4

    while need > 0:
        m = max(need * factor, 1024)
        x = np.random.uniform(lo, hi, size=(m, d))
        y = eval_expression(expr, x)
        keep = np.isfinite(y)
        x = x[keep]
        y = y[keep]

        if x.shape[0] == 0:
            factor = min(factor * 2, 1_000_000)
            continue

        take = min(need, x.shape[0])
        xs.append(x[:take])
        ys.append(y[:take])
        need -= take

    X = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, y


def sample_outer_band(n, d, lo, hi):
    mag = np.random.uniform(lo, hi, size=(n, d))
    sgn = np.where(np.random.rand(n, d) < 0.5, -1.0, 1.0)
    return sgn * mag


def sample_dataset_test_outer(expr, d, n):
    xs = []
    ys = []
    need = n
    factor = 4

    while need > 0:
        m = max(need * factor, 1024)
        x = sample_outer_band(m, d, lo=DataCFG.test_domain_min_abs, hi=DataCFG.test_domain_max_abs)
        y = eval_expression(expr, x)
        keep = np.isfinite(y)
        x = x[keep]
        y = y[keep]

        if x.shape[0] == 0:
            factor = min(factor * 2, 1_000_000)
            continue

        take = min(need, x.shape[0])
        xs.append(x[:take])
        ys.append(y[:take])
        need -= take

    X = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, y


def write_str_dataset(g, name, s):
    dt = h5py.string_dtype(encoding="utf-8")
    if name in g:
        del g[name]
    g.create_dataset(name, data=np.array(s, dtype=dt))


def write_array(g, name, arr, compression="gzip", compression_opts=4):
    if name in g:
        del g[name]
    g.create_dataset(
        name,
        data=arr,
        compression=compression,
        compression_opts=compression_opts,
        shuffle=True,
        chunks=True,
    )


if __name__ == "__main__":
    expr = generate_expression(k=DataCFG.k)

    with h5py.File(DataCFG.out_path, "w") as f:
        f.attrs["seed"] = int(DataCFG.seed)
        f.attrs["k"] = int(DataCFG.k)

        f.attrs["train_domain_min"] = float(DataCFG.train_domain_min)
        f.attrs["train_domain_max"] = float(DataCFG.train_domain_max)
        f.attrs["test_domain_min_abs"] = float(DataCFG.test_domain_min_abs)
        f.attrs["test_domain_max_abs"] = float(DataCFG.test_domain_max_abs)

        f.attrs["n_train"] = int(DataCFG.n_train)
        f.attrs["n_test"] = int(DataCFG.n_test)
        f.attrs["d_list"] = np.array(list(DataCFG.d_list), dtype=np.int32)

        g_expr = f.require_group("expressions")
        g_data = f.require_group("data")

        kind = expr["kind"]
        _, p_sym, y_sym = expr_to_sympy(expr)

        ge = g_expr.require_group(kind)
        ge.attrs["kind"] = kind
        ge.attrs["k"] = int(expr["k"])

        p_c0, p_lin, p_quad, p_cub, (ti, tj, tl) = expr["p"]
        ge.attrs["p_c0"] = float(p_c0)
        write_array(ge, "p_lin", np.asarray(p_lin, dtype=np.float32))
        write_array(ge, "p_quad", np.asarray(p_quad, dtype=np.float32))
        write_array(ge, "p_cub", np.asarray(p_cub, dtype=np.float32))
        write_array(ge, "p_cub_ti", np.asarray(ti, dtype=np.int32))
        write_array(ge, "p_cub_tj", np.asarray(tj, dtype=np.int32))
        write_array(ge, "p_cub_tl", np.asarray(tl, dtype=np.int32))

        write_str_dataset(ge, "sympy_P_srepr", sp.srepr(p_sym))
        write_str_dataset(ge, "sympy_y_srepr", sp.srepr(y_sym))
        write_str_dataset(ge, "sympy_P_str", str(p_sym))
        write_str_dataset(ge, "sympy_y_str", str(y_sym))

        gd_kind = g_data.require_group(kind)
        for d in DataCFG.d_list:
            X_train, y_train = sample_dataset_uniform(
                expr, d=d, n=DataCFG.n_train, lo=DataCFG.train_domain_min, hi=DataCFG.train_domain_max
            )
            X_test, y_test = sample_dataset_test_outer(expr, d=d, n=DataCFG.n_test)

            X_train = X_train.astype(np.float32, copy=False)
            y_train = y_train.astype(np.float32, copy=False)
            X_test = X_test.astype(np.float32, copy=False)
            y_test = y_test.astype(np.float32, copy=False)

            gd = gd_kind.require_group(f"d{d}")
            gt = gd.require_group("train")
            gs = gd.require_group("test")

            write_array(gt, "X", X_train)
            write_array(gt, "y", y_train)
            write_array(gs, "X", X_test)
            write_array(gs, "y", y_test)

    print("Saved:", DataCFG.out_path)


Saved: scalability_experiment_data.h5
